In [1]:
# If needed:
# !pip install -U openai

import os, json
from openai import OpenAI

# Make sure OPENAI_API_KEY is set in your environment.
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Pick a model that your account supports.
# Common choices: "gpt-4o-mini", "gpt-4.1-mini", etc.
MODEL = "gpt-4o-mini"

print("Using model:", MODEL)


Using model: gpt-4o-mini


In [2]:
def reactive_thermostat(temp_c: float, target: float = 22.0) -> str:
    r = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": (
                "You are a reactive thermostat agent. "
                "Rule: If temp < target respond ONLY with HEATER_ON, "
                "else respond ONLY with HEATER_OFF."
            )},
            {"role": "user", "content": f"temp={temp_c}, target={target}"}
        ]
    )
    return r.choices[0].message.content.strip()

print(reactive_thermostat(19))  # HEATER_ON
print(reactive_thermostat(25))  # HEATER_OFF


HEATER_ON
HEATER_OFF


In [3]:
def deliberative_route_choice(routes: list[dict]) -> str:
    # Step 1: Plan + score candidates (force JSON for clarity in class)
    plan = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": (
                "You are a deliberative agent. Create evaluation criteria and score each route. "
                "Output STRICT JSON only with keys: criteria, scored_routes "
                "(scored_routes is a list of {name, score, reason})."
            )},
            {"role": "user", "content": json.dumps({"routes": routes})}
        ]
    ).choices[0].message.content.strip()

    # Step 2: Use the plan output to choose
    final = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": (
                "Using the provided JSON analysis, choose the best route. "
                "Return exactly: Chosen: <name> | Reason: <one line>"
            )},
            {"role": "user", "content": plan}
        ]
    )
    return final.choices[0].message.content.strip()

routes = [
    {"name": "A", "distance_km": 12, "traffic": "high"},
    {"name": "B", "distance_km": 15, "traffic": "low"},
    {"name": "C", "distance_km": 13, "traffic": "medium"},
]

print(deliberative_route_choice(routes))


Chosen: B | Reason: It has the highest score of 5.0 due to low traffic and acceptable distance.


In [4]:
def goal_oriented_budget_agent(items: list[dict], budget: int) -> dict:
    '''
    items: [{"name": "...", "cost": 3000}, ...]
    budget: int
    '''
    state = {"items": items, "budget": budget}

    def total_cost(st):
        return sum(i["cost"] for i in st["items"])

    # Safety limit to avoid infinite loops in demos
    max_steps = 10
    steps = 0

    while total_cost(state) > budget and len(state["items"]) > 0 and steps < max_steps:
        r = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": (
                    "You are a goal-oriented budgeting agent. "
                    "Goal: bring total cost <= budget by removing exactly ONE item each step. "
                    "Return STRICT JSON only: {removed_item: <name>, remaining_items: [...]} "
                    "Choose the best item to remove (typically highest cost)."
                )},
                {"role": "user", "content": json.dumps(state)}
            ]
        )

        step = json.loads(r.choices[0].message.content)
        state["items"] = step["remaining_items"]
        steps += 1
        print(f"Step {steps} | Removed: {step['removed_item']} | Total now: {total_cost(state)}")

    return state

items = [
    {"name": "Shoes", "cost": 3000},
    {"name": "Groceries", "cost": 2500},
    {"name": "Dinner", "cost": 1800},
    {"name": "Taxi", "cost": 900},
]

final_state = goal_oriented_budget_agent(items, budget=6000)
print("Final items:", final_state["items"])
print("Final total:", sum(i["cost"] for i in final_state["items"]))


Step 1 | Removed: Shoes | Total now: 5200
Final items: [{'name': 'Groceries', 'cost': 2500}, {'name': 'Dinner', 'cost': 1800}, {'name': 'Taxi', 'cost': 900}]
Final total: 5200


In [5]:
# --- Example tools (replace with real API/DB calls in production) ---
def get_account_balance(user_id: str) -> dict:
    return {"user_id": user_id, "balance_inr": 125430.50}

def get_recent_transactions(user_id: str) -> dict:
    return {
        "user_id": user_id,
        "txns": [
            {"id": "T1", "amount": 25000, "merchant": "ABC Store", "country": "SG"},
            {"id": "T2", "amount": 900, "merchant": "Taxi", "country": "IN"},
        ]
    }

TOOL_MAP = {
    "get_account_balance": get_account_balance,
    "get_recent_transactions": get_recent_transactions,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_account_balance",
            "description": "Fetch account balance for a user",
            "parameters": {
                "type": "object",
                "properties": {"user_id": {"type": "string"}},
                "required": ["user_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_recent_transactions",
            "description": "Fetch recent transactions for a user",
            "parameters": {
                "type": "object",
                "properties": {"user_id": {"type": "string"}},
                "required": ["user_id"],
            },
        },
    },
]


In [6]:
def get_account_balance(user_id: str) -> dict:
    return {"user_id": user_id, "balance_inr": 125430.50}

TOOL_MAP = {"get_account_balance": get_account_balance}

def tool_using_agent(user_query: str) -> str:
    fallback = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": (
                "Tools are NOT available. Return STRICT JSON only: "
                "{tool: <tool_name>, args: {...}}. "
                "Allowed tool: get_account_balance."
            )},
            {"role": "user", "content": user_query}
        ]
    ).choices[0].message.content.strip()

    req = json.loads(fallback)
    tool_name = req["tool"]
    args = req["args"]
    tool_result = TOOL_MAP[tool_name](**args)

    final = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "Answer the user using the tool result."},
            {
                "role": "user",
                "content": (
                    f"User question: {user_query}\n"
                    f"Tool result: {json.dumps(tool_result)}"
                )
            }
        ]
    )
    return final.choices[0].message.content.strip()

print(tool_using_agent("What is the balance for user_id=U123?"))


The balance for user_id U123 is ₹125,430.50.


In [7]:
def agent(role_system_prompt: str, user_content: str, temperature: float = 0) -> str:
    r = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": role_system_prompt},
            {"role": "user", "content": user_content},
        ]
    )
    return r.choices[0].message.content.strip()

def multi_agent_run(task: str) -> str:
    plan = agent(
        "You are the Planner. Create a step-by-step plan only. No extra text.",
        task
    )

    execution = agent(
        "You are the Executor. Execute the plan and produce a draft output.",
        f"TASK:\n{task}\n\nPLAN:\n{plan}"
    )

    validation = agent(
        "You are the Validator. Check for errors, missing steps, assumptions. Return bullet fixes only.",
        f"TASK:\n{task}\n\nDRAFT OUTPUT:\n{execution}"
    )

    final = agent(
        "You are the Coordinator. Produce the final answer by applying validator fixes.",
        f"TASK:\n{task}\n\nPLAN:\n{plan}\n\nDRAFT:\n{execution}\n\nVALIDATION FIXES:\n{validation}"
    )

    return final

print(multi_agent_run("Draft a fraud investigation summary for transaction TX101 and recommend next steps."))


### Fraud Investigation Summary for Transaction TX101

**Transaction Details:**
- **Transaction ID:** TX101
- **Date:** October 15, 2023
- **Amount:** $10,500
- **Parties Involved:** John Doe (Payer), ABC Corp (Payee)
- **Payment Method:** Wire Transfer

**Transaction History Review:**
Upon analyzing the transaction history of the involved parties, several unusual patterns were identified:
- John Doe has made multiple transactions in the past, but none exceeded $5,000.
- ABC Corp has not received any transactions over $7,000 in the last six months.

**Red Flags Identified:**
During the review, the following red flags were noted:
- Mismatched addresses between John Doe (123 Elm St, Springfield) and ABC Corp (456 Oak St, Springfield).
- The transaction amount of $10,500 was significantly higher than previous transactions.
- A series of rapid transactions occurring within a short time frame, with three transactions made by John Doe within 24 hours.

**Interviews Conducted:**
Interviews we